In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import pickle
import tqdm
import math

pd.set_option('display.max_columns',50)

In [2]:
teams = pd.read_csv('data/MTeams.csv')
spellings = pd.read_csv('data/MTeamSpellings.csv')
model = pickle.load(open('model/lgbm_modelM.pkl', 'rb'))
team_data = pd.read_csv('data/team_data.csv')
submission = pd.read_csv('./data/SampleSubmissionStage2.csv')

In [3]:
def normal_cdf(x):
    # Vectorize math.erf so it can handle array inputs
    erf_vectorized = np.vectorize(lambda t: math.erf(t))
    return 0.5 * (1 + erf_vectorized(x / np.sqrt(2)))

def margin_to_probability(margin, scale=10.0):
    """
    Convert predicted margin (scalar or array) to win probability for Team A
    using a Normal CDF approximation.
    """
    return normal_cdf(margin / scale)

def get_preds(TeamID_A, TeamID_B):
    feature_cols = ['ORPerc_diff_7', 'TOPerc_diff_7', 'DefEff_diff_7', 'EffFG_diff_7', 'FTPerc_diff_7']
    data_team_A = team_data[team_data['Team'] == TeamID_A][feature_cols].tail(1).reset_index(drop=True).add_suffix('_A')
    data_team_B = team_data[team_data['Team'] == TeamID_B][feature_cols].tail(1).reset_index(drop=True).add_suffix('_B')
    
    pred_data = pd.concat([data_team_A, data_team_B], axis=1)
    preds_margin = model.predict(pred_data)
    preds_prob = margin_to_probability(preds_margin)
    return preds_prob
# Create a dictionary mapping team name spellings to Team IDs
team_map = dict(zip(teams['TeamID'], teams['TeamName']))
team_seo_map = dict(zip(spellings['TeamNameSpelling'], spellings['TeamID']))

In [4]:
preds_list = []
for row in tqdm.tqdm(submission.itertuples()):
    year = int(row.ID.split('_')[0])
    teamA_ID = int(row.ID.split('_')[1])
    teamB_ID = int(row.ID.split('_')[2])
    preds = dict()
    if teamA_ID < 3000:
        preds['year'] = year
        preds['teamA_ID'] = teamA_ID
        preds['teamB_ID'] = teamB_ID

        preds['win_probA'] = get_preds(teamA_ID, teamB_ID)[0]
        preds_list.append(preds)
    else:
        break

66066it [02:07, 518.11it/s]


In [5]:
preds_df = pd.DataFrame(preds_list, columns=['year','teamA_ID','teamB_ID','win_probA']).dropna()
preds_df.head()

,year,teamA_ID,teamB_ID,win_probA
0,2025,1101,1102,0.969173
1,2025,1101,1103,0.437453
2,2025,1101,1104,0.488222
3,2025,1101,1105,0.836987
4,2025,1101,1106,0.408826


In [6]:
preds_df.tail()

,year,teamA_ID,teamB_ID,win_probA
66061,2025,1477,1479,0.245575
66062,2025,1477,1480,0.522565
66063,2025,1478,1479,0.316729
66064,2025,1478,1480,0.546615
66065,2025,1479,1480,0.754520


In [7]:
final_team_data = pd.merge(teams[['TeamID', 'TeamName']], spellings, on='TeamID')
final_team_data.head()

,TeamID,TeamName,TeamNameSpelling
0,1101,Abilene Chr,abilene chr
1,1101,Abilene Chr,abilene christian
2,1101,Abilene Chr,abilene-christian
3,1102,Air Force,air force
4,1102,Air Force,air-force


In [8]:
complete_preds = pd.merge(preds_df, final_team_data, left_on='teamA_ID', right_on='TeamID', suffixes=('','_A'))
complete_preds = pd.merge(complete_preds, final_team_data, left_on='teamB_ID', right_on='TeamID', suffixes=('','_B'))
complete_preds

,year,teamA_ID,teamB_ID,win_probA,TeamID,TeamName,TeamNameSpelling,TeamID_B,TeamName_B,TeamNameSpelling_B
0,2025,1101,1102,0.969173,1101,Abilene Chr,abilene chr,1102,Air Force,air force
1,2025,1101,1102,0.969173,1101,Abilene Chr,abilene chr,1102,Air Force,air-force
2,2025,1101,1102,0.969173,1101,Abilene Chr,abilene christian,1102,Air Force,air force
3,2025,1101,1102,0.969173,1101,Abilene Chr,abilene christian,1102,Air Force,air-force
4,2025,1101,1102,0.969173,1101,Abilene Chr,abilene-christian,1102,Air Force,air force
...,...,...,...,...,...,...,...,...,...,...
647154,2025,1477,1480,0.522565,1477,East Texas A&M,texas a&m-commerce,1480,West Georgia,west georgia
647155,2025,1477,1480,0.522565,1477,East Texas A&M,tx a&m commerce,1480,West Georgia,west georgia
647156,2025,1478,1479,0.316729,1478,Le Moyne,le moyne,1479,Mercyhurst,mercyhurst
647157,2025,1478,1480,0.546615,1478,Le Moyne,le moyne,1480,West Georgia,west georgia


In [9]:
complete_preds.to_csv('data/final_complete_preds.csv', index=False)